# Methanol Price Prediction - Real Data Version
Reid Powning

This notebook replaces the synthetic dataset from the first version with **real pulled data**:
* Feature series (natural gas, crude oil, USD index, housing starts, China industrial production, gasoline) pulled live from FRED
* Target series (methanol price) from Methanex's historical posted price file

**Before running this notebook (in Google Colab):**
1. Go to methanex.com/about-methanol/pricing and download the *Historical Methanex Posted Price* Excel file.
2. In Colab, click the folder icon in the left sidebar, then the upload icon, and upload that file. Rename it to `methanex_historical_price.xlsx` if it isn't already, so it matches the filename used below.
3. Click the key icon (🔑) in the left sidebar → Add new secret → name it `FRED_API_KEY` → paste your key as the value → toggle notebook access ON.

Files uploaded this way live only for the current Colab session - if your runtime disconnects or you come back tomorrow, you'll need to re-upload the Excel file (the FRED_API_KEY secret persists though, since it's tied to your Google account, not the session).

### 1) Set up the FRED connection

In [104]:
# Colab needs the ! prefix to run pip as a shell command, not a Python import
!pip install fredapi -q

# pandas: for loading/manipulating tabular data
import pandas as pd

# fredapi: the official-style Python wrapper for FRED's API
from fredapi import Fred

# google.colab.userdata: reads secrets from Colab's Secrets panel instead of
# pasting the raw key into this notebook's cells
from google.colab import userdata

fred = Fred(api_key=userdata.get('FRED_API_KEY'))

### 2) Pull the raw feature series from FRED
Each `fred.get_series()` call returns the FULL history of that series as a pandas Series, indexed by date. These are pulled at their native frequency (daily/weekly/monthly) - we'll line them all up to monthly in the next step.

In [105]:
# Henry Hub Natural Gas Spot Price - daily, $/MMBtu - the main feedstock cost driver
natgas = fred.get_series("DHHNGSP")

# WTI Crude Oil Spot Price - daily, $/bbl - transport cost / petrochemical sentiment proxy
crude = fred.get_series("DCOILWTICO")

# Trade Weighted US Dollar Index (Broad) - daily - commodities are USD-denominated
usd_index = fred.get_series("DTWEXBGS")

# Housing Starts - monthly, thousands of units - formaldehyde/resin demand proxy
housing_starts = fred.get_series("HOUST")

# China Industrial Production Index (OECD Main Economic Indicators, via FRED) -
# used as the China demand proxy instead of PMI, since FRED doesn't carry an official
# China Manufacturing PMI series directly (it's an NBS China release, not one FRED mirrors)
china_industrial_production = fred.get_series("CHNPRINTO01IXPYM")

# US Regular Gasoline Price - weekly, $/gal - MTBE/fuel-blending demand proxy
gasoline = fred.get_series("GASREGW")

# quick sanity check - print the last few values of each series
print("Natural gas (last 5):\n", natgas.tail())
print("\nCrude oil (last 5):\n", crude.tail())

Natural gas (last 5):
 2026-08-26    2.81
2026-08-27    2.89
2026-08-28    2.82
2026-08-31    2.90
2026-09-01    2.90
dtype: float64

Crude oil (last 5):
 2026-08-26    83.46
2026-08-27    84.81
2026-08-28    84.57
2026-08-31    87.03
2026-09-01    91.48
dtype: float64


### 3) Resample everything to monthly
Different series come from FRED at different frequencies. Since our target (Methanex posted price) is monthly, we resample every feature to monthly using `.resample("MS").mean()` - "MS" means "month start", so every series ends up indexed on the first of each month, which lets us merge them cleanly on that shared date index.

In [106]:
natgas_m = natgas.resample("MS").mean()
crude_m = crude.resample("MS").mean()
usd_index_m = usd_index.resample("MS").mean()
gasoline_m = gasoline.resample("MS").mean()
# housing starts and China industrial production are usually already monthly, but we resample
# anyway just to guarantee the index lines up exactly on month-start dates like the other series
housing_starts_m = housing_starts.resample("MS").mean()
china_industrial_production_m = china_industrial_production.resample("MS").mean()

In [107]:
# merge every resampled series into one dataframe, aligned on the shared monthly Date index
fred_features = pd.DataFrame({
    "NaturalGasPrice": natgas_m,
    "CrudeOilPrice": crude_m,
    "USDIndex": usd_index_m,
    "HousingStarts": housing_starts_m,
    "ChinaIndustrialProduction": china_industrial_production_m,
    "GasolinePrice": gasoline_m,
})

# drop any month where at least one series doesn't have data yet (usually just the most recent
# month or two, since some series report with a lag)
fred_features = fred_features.dropna()
fred_features.index.name = "Date"
fred_features.tail()

,NaturalGasPrice,CrudeOilPrice,USDIndex,HousingStarts,ChinaIndustrialProduction,GasolinePrice
Date,,,,,,
2023-07-01,2.551000,76.069500,118.049165,1464.0,103.7,3.59700
2023-08-01,2.583043,81.386087,119.856348,1321.0,104.5,3.83975
2023-09-01,2.636500,89.425000,121.645530,1369.0,104.5,3.83600
2023-10-01,2.981429,85.639524,123.497148,1380.0,104.6,3.61280
2023-11-01,2.707619,77.685000,121.222120,1522.0,106.6,3.31800


### 4) Load the Methanex posted price (the target variable)
This is the file you uploaded to Colab's file browser. Methanex's historical file typically has separate columns per region (Europe / North America / Asia Pacific / China), so we melt it from wide format (one column per region) into long format (one row per Region + Date), which also happens to give us a `Region` column for free - the same categorical feature we one-hot encoded in the synthetic-data version.

**Note:** the exact column names in Methanex's file may not match the placeholders below exactly - open the Excel file first with `pd.read_excel("methanex_historical_price.xlsx").head()` to see the real column names, then adjust the `rename()` mapping in the next cell to match.

In [108]:
# load the raw Methanex file - adjust the filename if yours is named differently
methanex_raw = pd.read_excel("methanex_historical_price.xlsx")

# inspect the actual column names/structure before doing anything else - Methanex's exact
# layout can shift between file versions, so always check this first
methanex_raw.head(10)

/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,Methanex Monthly Average Regional Posted Contract Price History,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Methanex Non-Discounted Reference Price,NaN,Methanex European Posted Contract Price,Methanex Asian Posted Contract Price,Methanex China Posted Contract Price
3,NaN,(MNDRP),NaN,(MEPCP),(APCP),(CPCP)
4,Date,$/gal,$/MT,€/MT,$/MT,$/MT
5,2001-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN
6,2001-02-01 00:00:00,NaN,NaN,NaN,NaN,NaN
7,2001-03-01 00:00:00,NaN,NaN,NaN,NaN,NaN
8,2001-04-01 00:00:00,NaN,NaN,NaN,NaN,NaN
9,2001-05-01 00:00:00,0.77,256.102,NaN,NaN,NaN


In [109]:
date_col_name = "Methanex Monthly Average Regional Posted Contract Price History"

# Map the 'Unnamed' columns to the desired region names based on the content in the raw data
region_col_mapping = {
    "Unnamed: 2": "North America", # MNDRP seems to refer to North America
    "Unnamed: 3": "Europe",
    "Unnamed: 4": "Asia Pacific",
    "Unnamed: 5": "China",
}

# Combine into a single rename dictionary
rename_dict = {date_col_name: "Date", **region_col_mapping}

# Apply rename to the columns
methanex_raw = methanex_raw.rename(columns=rename_dict)

# Drop the first 5 rows which contain metadata/malformed headers
methanex_raw = methanex_raw.iloc[5:].copy()

# make sure Date is an actual datetime and snapped to month-start so it merges cleanly
methanex_raw["Date"] = pd.to_datetime(methanex_raw["Date"]).values.astype("datetime64[M]")

# The value_vars for melt should now be the new, desired region names
melt_value_vars = list(region_col_mapping.values())

methanol_price_long = methanex_raw.melt(
    id_vars=["Date"],
    value_vars=melt_value_vars,
    var_name="Region",
    value_name="MethanolPrice"
)

methanol_price_long = methanol_price_long.dropna(subset=["MethanolPrice"])
methanol_price_long.head()

,Date,Region,MethanolPrice
4,2001-05-01,North America,256.102
5,2001-06-01,North America,222.842
6,2001-07-01,North America,189.582
7,2001-08-01,North America,169.626
8,2001-09-01,North America,139.692


### 5) Merge target + features into one dataset
Join the long-format methanol price data to the monthly FRED features on `Date`. Since every region shares the same macro features (natural gas, crude, USD, etc. aren't region-specific in this feature set), each region's row for a given month gets the same feature values, similar to how the synthetic dataset was structured.

In [110]:
methanoldata = methanol_price_long.merge(fred_features, on="Date", how="inner")

# FeedstockType wasn't pulled from FRED - add it back in as a simple mapping based on Region,
# same assumption as the synthetic dataset (China leans coal-based, everyone else natural gas)
region_feedstock_map = {
    "North America": "Natural Gas",
    "Europe": "Natural Gas",
    "Asia Pacific": "Natural Gas",
    "China": "Coal",
}
methanoldata["FeedstockType"] = methanoldata["Region"].map(region_feedstock_map)

# Month column for seasonality, same as the synthetic version
methanoldata["Month"] = methanoldata["Date"].dt.month

methanoldata.head()

,Date,Region,MethanolPrice,NaturalGasPrice,CrudeOilPrice,USDIndex,HousingStarts,ChinaIndustrialProduction,GasolinePrice,FeedstockType,Month
0,2006-02-01,North America,355.882,7.536316,61.631053,100.211170,2119.0,120.1,2.28000,Natural Gas,2
1,2006-03-01,North America,355.882,6.888261,62.685217,100.428087,1969.0,117.8,2.42475,Natural Gas,3
2,2006-04-01,North America,355.882,7.163684,69.443684,99.743480,1821.0,116.6,2.74200,Natural Gas,4
3,2006-05-01,North America,342.578,6.245000,70.844091,97.511774,1942.0,117.9,2.90680,Natural Gas,5
4,2006-06-01,North America,342.578,6.210000,70.950909,98.692723,1802.0,119.5,2.88450,Natural Gas,6


In [153]:
# Add lagged MethanolPrice features, grouped by Region
lag_values = [1, 3, 12] # Lag for 1 month, 3 months, and 12 months

for lag in lag_values:
    methanoldata[f'MethanolPrice_lag_{lag}'] = methanoldata.groupby('Region')['MethanolPrice'].shift(lag)

# Fill NaN values created by shifting (e.g., for initial months of a region's data)
# Using 0 or the mean of the column are common strategies. For simplicity, we'll fill with 0 for now.
methanoldata = methanoldata.fillna(0)

methanoldata.head()

,Date,Region,MethanolPrice,NaturalGasPrice,CrudeOilPrice,USDIndex,HousingStarts,ChinaIndustrialProduction,GasolinePrice,FeedstockType,Month,MethanolPrice_lag_1,MethanolPrice_lag_3,MethanolPrice_lag_12
0,2006-02-01,North America,355.882,7.536316,61.631053,100.211170,2119.0,120.1,2.28000,Natural Gas,2,0.000,0.000,0.0
1,2006-03-01,North America,355.882,6.888261,62.685217,100.428087,1969.0,117.8,2.42475,Natural Gas,3,355.882,0.000,0.0
2,2006-04-01,North America,355.882,7.163684,69.443684,99.743480,1821.0,116.6,2.74200,Natural Gas,4,355.882,0.000,0.0
3,2006-05-01,North America,342.578,6.245000,70.844091,97.511774,1942.0,117.9,2.90680,Natural Gas,5,355.882,355.882,0.0
4,2006-06-01,North America,342.578,6.210000,70.950909,98.692723,1802.0,119.5,2.88450,Natural Gas,6,342.578,355.882,0.0


In [154]:
# After adding lagged features, check for duplicates again and drop if any
print(f"Duplicates before dropping: {methanoldata.duplicated().sum()}")
methanoldata = methanoldata.drop_duplicates()
print(f"Number of rows after dropping duplicates: {methanoldata.shape[0]}")

# Drop any rows with nulls that might have been introduced by lagging not handled by fillna(0) or other issues
methanoldata = methanoldata.dropna()

# Define features dataframe, now including the new lagged features
methanol_features_df = methanoldata[[
    "Region",
    "FeedstockType",
    "NaturalGasPrice",
    "CrudeOilPrice",
    "USDIndex",
    "HousingStarts",
    "ChinaIndustrialProduction",
    "GasolinePrice",
    "MethanolPrice_lag_1",
    "MethanolPrice_lag_3",
    "MethanolPrice_lag_12",
]]

# Define the label dataframe
methanol_price_df = methanoldata[["MethanolPrice"]]

methanol_features_df.head()

Duplicates before dropping: 0
Number of rows after dropping duplicates: 613


,Region,FeedstockType,NaturalGasPrice,CrudeOilPrice,USDIndex,HousingStarts,ChinaIndustrialProduction,GasolinePrice,MethanolPrice_lag_1,MethanolPrice_lag_3,MethanolPrice_lag_12
0,North America,Natural Gas,7.536316,61.631053,100.211170,2119.0,120.1,2.28000,0.000,0.000,0.0
1,North America,Natural Gas,6.888261,62.685217,100.428087,1969.0,117.8,2.42475,355.882,0.000,0.0
2,North America,Natural Gas,7.163684,69.443684,99.743480,1821.0,116.6,2.74200,355.882,0.000,0.0
3,North America,Natural Gas,6.245000,70.844091,97.511774,1942.0,117.9,2.90680,355.882,355.882,0.0
4,North America,Natural Gas,6.210000,70.950909,98.692723,1802.0,119.5,2.88450,342.578,355.882,0.0


In [155]:
# one-hot encode Region and FeedstockType, same pattern as before
from sklearn.preprocessing import OneHotEncoder

region_list_of_lists = [[r] for r in methanol_features_df["Region"].tolist()]
feedstock_list_of_lists = [[f] for f in methanol_features_df["FeedstockType"].tolist()]

regionencoder = OneHotEncoder()
feedstockencoder = OneHotEncoder()

regionencoder.fit(region_list_of_lists)
feedstockencoder.fit(feedstock_list_of_lists)

regionencodertransformed = regionencoder.transform(region_list_of_lists).toarray()
feedstockencodertransformed = feedstockencoder.transform(feedstock_list_of_lists).toarray()

regiontransformed_df = pd.DataFrame(
    regionencodertransformed,
    columns=[f"Region_{cat}" for cat in regionencoder.categories_[0]]
)
feedstocktransformed_df = pd.DataFrame(
    feedstockencodertransformed,
    columns=[f"Feedstock_{cat}" for cat in feedstockencoder.categories_[0]]
)

regiontransformed_df.reset_index(drop=True, inplace=True)
feedstocktransformed_df.reset_index(drop=True, inplace=True)
methanol_features_df = methanol_features_df.reset_index(drop=True)

methanol_features_transformed_df = pd.concat(
    [methanol_features_df, regiontransformed_df, feedstocktransformed_df],
    axis=1
).drop(["Region", "FeedstockType"], axis=1)

methanol_features_transformed_df.shape

(613, 15)

In [156]:
# scale the numeric columns, now including lagged features
from sklearn.preprocessing import StandardScaler as ss

methanolscaler = ss()
numeric_cols = [
    "NaturalGasPrice", "CrudeOilPrice", "USDIndex",
    "HousingStarts", "ChinaIndustrialProduction", "GasolinePrice",
    "MethanolPrice_lag_1", "MethanolPrice_lag_3", "MethanolPrice_lag_12",
]
methanol_features_transformed_df[numeric_cols] = methanolscaler.fit_transform(
    methanol_features_transformed_df[numeric_cols]
)
methanol_features_transformed_df.head()

,NaturalGasPrice,CrudeOilPrice,USDIndex,HousingStarts,ChinaIndustrialProduction,GasolinePrice,MethanolPrice_lag_1,MethanolPrice_lag_3,MethanolPrice_lag_12,Region_Asia Pacific,Region_China,Region_Europe,Region_North America,Feedstock_Coal,Feedstock_Natural Gas
0,1.659324,-0.523655,-0.348127,2.697896,2.215655,-1.089437,-3.459337,-3.184723,-2.42201,0.0,0.0,0.0,1.0,0.0,1.0
1,1.340615,-0.476340,-0.329669,2.288980,1.753150,-0.861602,-0.281903,-3.184723,-2.42201,0.0,0.0,0.0,1.0,0.0,1.0
2,1.476066,-0.172991,-0.387924,1.885515,1.511843,-0.362252,-0.281903,-3.184723,-2.42201,0.0,0.0,0.0,1.0,0.0,1.0
3,1.024264,-0.110135,-0.577826,2.215375,1.773259,-0.102858,-0.281903,-0.219866,-2.42201,0.0,0.0,0.0,1.0,0.0,1.0
4,1.007051,-0.105340,-0.477336,1.833719,2.095002,-0.137958,-0.400685,-0.219866,-2.42201,0.0,0.0,0.0,1.0,0.0,1.0


In [157]:
# chronological train/test split, same reasoning as before: this is time-series data,
# so we train on the earliest months and test on the most recent months rather than
# randomly shuffling (which would leak future information into training)
methanol_features_transformed_df["Date"] = methanoldata["Date"].reset_index(drop=True)
methanol_price_df = methanol_price_df.reset_index(drop=True)

sort_order = methanol_features_transformed_df["Date"].argsort()
X = methanol_features_transformed_df.iloc[sort_order].drop(columns=["Date"]).reset_index(drop=True)
y = methanol_price_df.iloc[sort_order].reset_index(drop=True)

split_index = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Train rows: {X_train.shape[0]}, Test rows: {X_test.shape[0]}")

Train rows: 490, Test rows: 123


In [158]:
# fit the baseline linear regression, same as before
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: ${mae:.2f} per MT")
print(f"R-squared: {r2:.3f}")

Mean Absolute Error: $29.96 per MT
R-squared: 0.833


In [140]:
# inspect feature coefficients, same as before
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": model.coef_[0]
}).sort_values("Coefficient", key=abs, ascending=False)

coef_df

,Feature,Coefficient
6,MethanolPrice_lag_1,9.516195e+01
1,CrudeOilPrice,3.795401e+01
3,HousingStarts,1.810220e+01
11,Region_Europe,-1.780325e+01
5,GasolinePrice,-1.777391e+01
7,MethanolPrice_lag_3,-1.658696e+01
12,Region_North America,1.405336e+01
0,NaturalGasPrice,-8.152616e+00
4,ChinaIndustrialProduction,-4.250596e+00
9,Region_Asia Pacific,3.749890e+00


In [141]:
# save the merged real dataset out to CSV so you have a permanent snapshot,
# and so the rest of this notebook can be re-run without re-hitting the FRED API every time
methanoldata.to_csv("methanol_data_real.csv", index=False)
methanoldata.shape

(613, 14)

### 6) From here, the pipeline is IDENTICAL to the synthetic-data notebook
Same duplicate check, same feature/label split, same one-hot encoding of Region/FeedstockType, same StandardScaler, same chronological train/test split, same LinearRegression baseline. The only thing that changed is where the numbers came from.

In [142]:
# duplicate check (same as before)
print(methanoldata.duplicated().sum())
methanoldata = methanoldata.drop_duplicates()
methanoldata.shape[0]

0


613

In [143]:
# drop any rows with nulls
methanoldata = methanoldata.dropna()

# define features dataframe - same 8 columns as the synthetic version
methanol_features_df = methanoldata[[
    "Region",
    "FeedstockType",
    "NaturalGasPrice",
    "CrudeOilPrice",
    "USDIndex",
    "HousingStarts",
    "ChinaIndustrialProduction",
    "GasolinePrice",
]]

# define the label dataframe
methanol_price_df = methanoldata[["MethanolPrice"]]
methanol_features_df.head()

,Region,FeedstockType,NaturalGasPrice,CrudeOilPrice,USDIndex,HousingStarts,ChinaIndustrialProduction,GasolinePrice
0,North America,Natural Gas,7.536316,61.631053,100.211170,2119.0,120.1,2.28000
1,North America,Natural Gas,6.888261,62.685217,100.428087,1969.0,117.8,2.42475
2,North America,Natural Gas,7.163684,69.443684,99.743480,1821.0,116.6,2.74200
3,North America,Natural Gas,6.245000,70.844091,97.511774,1942.0,117.9,2.90680
4,North America,Natural Gas,6.210000,70.950909,98.692723,1802.0,119.5,2.88450


In [144]:
methanoldata.shape[0]

613

In [145]:
# one-hot encode Region and FeedstockType, same pattern as before
from sklearn.preprocessing import OneHotEncoder

region_list_of_lists = [[r] for r in methanol_features_df["Region"].tolist()]
feedstock_list_of_lists = [[f] for f in methanol_features_df["FeedstockType"].tolist()]

regionencoder = OneHotEncoder()
feedstockencoder = OneHotEncoder()

regionencoder.fit(region_list_of_lists)
feedstockencoder.fit(feedstock_list_of_lists)

regionencodertransformed = regionencoder.transform(region_list_of_lists).toarray()
feedstockencodertransformed = feedstockencoder.transform(feedstock_list_of_lists).toarray()

regiontransformed_df = pd.DataFrame(
    regionencodertransformed,
    columns=[f"Region_{cat}" for cat in regionencoder.categories_[0]]
)
feedstocktransformed_df = pd.DataFrame(
    feedstockencodertransformed,
    columns=[f"Feedstock_{cat}" for cat in feedstockencoder.categories_[0]]
)

regiontransformed_df.reset_index(drop=True, inplace=True)
feedstocktransformed_df.reset_index(drop=True, inplace=True)
methanol_features_df = methanol_features_df.reset_index(drop=True)

methanol_features_transformed_df = pd.concat(
    [methanol_features_df, regiontransformed_df, feedstocktransformed_df],
    axis=1
).drop(["Region", "FeedstockType"], axis=1)

methanol_features_transformed_df.shape

(613, 12)

In [146]:
# scale the numeric columns, same as before
from sklearn.preprocessing import StandardScaler as ss

methanolscaler = ss()
numeric_cols = [
    "NaturalGasPrice", "CrudeOilPrice", "USDIndex",
    "HousingStarts", "ChinaIndustrialProduction", "GasolinePrice",
]
methanol_features_transformed_df[numeric_cols] = methanolscaler.fit_transform(
    methanol_features_transformed_df[numeric_cols]
)
methanol_features_transformed_df.head()

,NaturalGasPrice,CrudeOilPrice,USDIndex,HousingStarts,ChinaIndustrialProduction,GasolinePrice,Region_Asia Pacific,Region_China,Region_Europe,Region_North America,Feedstock_Coal,Feedstock_Natural Gas
0,1.659324,-0.523655,-0.348127,2.697896,2.215655,-1.089437,0.0,0.0,0.0,1.0,0.0,1.0
1,1.340615,-0.476340,-0.329669,2.288980,1.753150,-0.861602,0.0,0.0,0.0,1.0,0.0,1.0
2,1.476066,-0.172991,-0.387924,1.885515,1.511843,-0.362252,0.0,0.0,0.0,1.0,0.0,1.0
3,1.024264,-0.110135,-0.577826,2.215375,1.773259,-0.102858,0.0,0.0,0.0,1.0,0.0,1.0
4,1.007051,-0.105340,-0.477336,1.833719,2.095002,-0.137958,0.0,0.0,0.0,1.0,0.0,1.0


In [147]:
# chronological train/test split, same reasoning as before: this is time-series data,
# so we train on the earliest months and test on the most recent months rather than
# randomly shuffling (which would leak future information into training)
methanol_features_transformed_df["Date"] = methanoldata["Date"].reset_index(drop=True)
methanol_price_df = methanol_price_df.reset_index(drop=True)

sort_order = methanol_features_transformed_df["Date"].argsort()
X = methanol_features_transformed_df.iloc[sort_order].drop(columns=["Date"]).reset_index(drop=True)
y = methanol_price_df.iloc[sort_order].reset_index(drop=True)

split_index = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Train rows: {X_train.shape[0]}, Test rows: {X_test.shape[0]}")

Train rows: 490, Test rows: 123


In [148]:
# fit the baseline linear regression, same as before
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: ${mae:.2f} per MT")
print(f"R-squared: {r2:.3f}")

Mean Absolute Error: $69.78 per MT
R-squared: 0.391


In [149]:
# inspect feature coefficients, same as before
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": model.coef_[0]
}).sort_values("Coefficient", key=abs, ascending=False)

coef_df

,Feature,Coefficient
1,CrudeOilPrice,7.141591e+01
8,Region_Europe,-5.954415e+01
9,Region_North America,4.683339e+01
4,ChinaIndustrialProduction,-3.037663e+01
3,HousingStarts,3.013859e+01
6,Region_Asia Pacific,1.271076e+01
0,NaturalGasPrice,7.893613e+00
5,GasolinePrice,-5.312129e+00
2,USDIndex,2.887272e+00
7,Region_China,-2.442491e-15


### Notes on real-world results
Expect a much lower R² and higher MAE than the synthetic version's ~0.95 - real commodity price relationships are noisier than a formula generated on purpose to be predictable. That's normal, not a bug. If accuracy is disappointing, the usual next steps are: add lagged `MethanolPrice` columns (t-1, t-3, t-12 months), try a non-linear model (RandomForestRegressor), or extend the feature set with an actual coal price series instead of the natural-gas proxy used for China's coal-based feedstock cost.

## 7) Make Predictions with the Trained Model

Now that we have a trained model, we can use it to predict methanol prices based on new input data. Below, you can modify the input values to see how different economic factors and regional settings might influence the predicted methanol price.

In [160]:
import numpy as np

# --- User Input Fields ---
# Modify these values to make a new prediction

# Categorical Features
Region = "China" # @param ["North America","Europe","Asia Pacific","China"]
Feedstock_Type = "Natural Gas" # @param ["Natural Gas","Coal"]

# Numeric Features
Natural_Gas_Price = 3.0 # @param {"type":"number"}
Crude_Oil_Price = 70.0 # @param {"type":"number"}
USD_Index = 100.0 # @param {"type":"number"}
Housing_Starts = 1500.0 # @param {"type":"number"}
China_Industrial_Production_Index = 110.0 # @param {"type":"number"}
Gasoline_Price = 3.5 # @param {"type":"number"}
Methanol_PL_1Month = 300.0 # @param {"type":"number"}
Methanol_PL_3Months = 290.0 # @param {"type":"number"}
Methanol_PL_1Year = 280.0 # @param {"type":"number"}

# Create a DataFrame from the new inputs
# Ensure all numeric and categorical features are present, including lagged features
new_data = pd.DataFrame({
    "Region": [input_region],
    "FeedstockType": [input_feedstock_type],
    "NaturalGasPrice": [input_natural_gas_price],
    "CrudeOilPrice": [input_crude_oil_price],
    "USDIndex": [input_usd_index],
    "HousingStarts": [input_housing_starts],
    "ChinaIndustrialProduction": [input_china_industrial_production],
    "GasolinePrice": [input_gasoline_price],
    "MethanolPrice_lag_1": [input_methanol_price_lag_1],
    "MethanolPrice_lag_3": [input_methanol_price_lag_3],
    "MethanolPrice_lag_12": [input_methanol_price_lag_12]
})

# --- Apply the same preprocessing steps as the training data ---

# 1. One-hot encode 'Region' and 'FeedstockType' using the *fitted* encoders
# Ensure the input to transform is in the correct format (list of lists)
region_encoded = regionencoder.transform([[input_region]]).toarray()
feedstock_encoded = feedstockencoder.transform([[input_feedstock_type]]).toarray()

# Create DataFrames for the encoded features
region_df = pd.DataFrame(region_encoded, columns=[f"Region_{cat}" for cat in regionencoder.categories_[0]])
feedstock_df = pd.DataFrame(feedstock_encoded, columns=[f"Feedstock_{cat}" for cat in feedstockencoder.categories_[0]])

# Remove original categorical columns and concatenate encoded ones
new_data_processed = new_data.drop(columns=["Region", "FeedstockType"])
new_data_processed = pd.concat([new_data_processed.reset_index(drop=True), region_df, feedstock_df], axis=1)

# 2. Scale numeric features using the *fitted* StandardScaler
# The numeric_cols list must match what was used for fitting the scaler
scaling_cols = [
    "NaturalGasPrice", "CrudeOilPrice", "USDIndex",
    "HousingStarts", "ChinaIndustrialProduction", "GasolinePrice",
    "MethanolPrice_lag_1", "MethanolPrice_lag_3", "MethanolPrice_lag_12",
]

new_data_processed[scaling_cols] = methanolscaler.transform(new_data_processed[scaling_cols])

# Ensure the column order matches X_train.columns (critical for prediction)
# It is safest to reindex the new data to match the training data's column order
final_features_for_prediction = new_data_processed.reindex(columns=X_train.columns, fill_value=0)

# Make prediction
predicted_price = model.predict(final_features_for_prediction)[0][0]

print(f"Predicted Methanol Price: ${predicted_price:.2f} per MT")

Predicted Methanol Price: $330.68 per MT


In [161]:
# ---------------------------------------------------------------------
# One-shot export: everything needed to rebuild the dashboard's model
# ---------------------------------------------------------------------

print("=== COEFFICIENTS ===")
for feature, coef in zip(X_train.columns, model.coef_[0]):
    print(f'"{feature}": {coef},')

print("\n=== INTERCEPT ===")
print(model.intercept_[0])

print("\n=== SCALER MEAN / SCALE ===")
for col, mean, scale in zip(numeric_cols, methanolscaler.mean_, methanolscaler.scale_):
    print(f'{col}: {{ "mean": {mean}, "scale": {scale} }},')

print("\n=== NUMERIC FEATURE RANGES (real data) ===")
for col in numeric_cols:
    print(f'{col}: min={methanoldata[col].min()}, max={methanoldata[col].max()}')

print("\n=== MODEL PERFORMANCE ===")
print(f"MAE: {mae:.2f}")
print(f"R-squared: {r2:.3f}")
print(f"Train rows: {X_train.shape[0]}, Test rows: {X_test.shape[0]}")

=== COEFFICIENTS ===
"NaturalGasPrice": -8.152615501240694,
"CrudeOilPrice": 37.954009968986156,
"USDIndex": -2.980124431445077,
"HousingStarts": 18.102200077404017,
"ChinaIndustrialProduction": -4.250596196992532,
"GasolinePrice": -17.773911997433736,
"MethanolPrice_lag_1": 95.16195459184165,
"MethanolPrice_lag_3": -16.586956497564486,
"MethanolPrice_lag_12": -1.4724311831454902,
"Region_Asia Pacific": 3.749889883618989,
"Region_China": 3.552713678800501e-15,
"Region_Europe": -17.803247629442524,
"Region_North America": 14.053357745823542,
"Feedstock_Coal": 0.0,
"Feedstock_Natural Gas": 0.0,

=== INTERCEPT ===
388.6033043790388

=== SCALER MEAN / SCALE ===
NaturalGasPrice: { "mean": 4.162289329583914, "scale": 2.0333738269823805 },
CrudeOilPrice: { "mean": 73.29783387788281, "scale": 22.279501406943094 },
USDIndex: { "mean": 104.3023183766706, "scale": 11.751891203764695 },
HousingStarts: { "mean": 1129.3491027732464, "scale": 366.8232375258111 },
ChinaIndustrialProduction: { "mean": 